# Sampling Activations from R(2+1)D

Extracts layer activations from R(2+1)D-18 (Tran et al., CVPR 2018) using the same optical-flow stimuli as `s01_sample_fnn.ipynb`. R(2+1)D uses factored (2D spatial + 1D temporal) convolutions, giving a growing temporal receptive field across layers.

Install: already in `torchvision.models.video` — no extra install needed.

Output: `data/sampled/tensor4d_r2plus1d_<LAYER>_i3_n<N>_seed17.npy` (shape: neurons × 11 × 8 × T') where T' decreases with depth (37, 19, 10, 5 for layers 1–4).

In [28]:
import numpy as np
import torch
import torch.nn.functional as F
import sys, os
_d = os.path.abspath(os.getcwd())
sys.path.insert(0, _d if os.path.isdir(os.path.join(_d, 'src')) else os.path.dirname(_d))
from src.plot_utils import createFlowDataset
from time import time

from torchvision.models.video import r2plus1d_18, R2Plus1D_18_Weights

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

######################## PARAMS ########################
LAYERS = ['layer1', 'layer2', 'layer3', 'layer4']
n_fmaps_to_sample = 40      # max feature maps (channels) to sample per layer
samples_per_fmap  = 50      # max spatial positions sampled per feature map
seed              = 17
N_INSTANCES       = 3
trial_len         = 37      # frames per stimulus trial
NDIRS             = 8
scl_factor        = 0.7
R2PLUS1D_SIZE     = 112     # spatial input size (model default)

# Kinetics-400 normalisation constants (R(2+1)D pretrained on Kinetics)
KINETICS_MEAN = np.array([0.43216, 0.394666, 0.37645], dtype='float32')
KINETICS_STD  = np.array([0.22803, 0.22145, 0.216989], dtype='float32')

# Optical-flow stimulus parameters (must match nb01)
topdir      = '../stimuli/flowstims'
orig_shape  = (800, 600)
input_shape = (112, 112)
mydirs      = ['0', '45', '90', '135', '180', '225', '270', '315']
categories  = [
    'grat_W12', 'grat_W1', 'grat_W2',
    'neg1dotflow_D1_bg', 'neg1dotflow_D2_bg',
    'neg3dotflow_D1_bg', 'neg3dotflow_D2_bg',
    'pos1dotflow_D1_bg', 'pos1dotflow_D2_bg',
    'pos3dotflow_D1_bg', 'pos3dotflow_D2_bg',
]
NSTIMS = len(categories)
assert NSTIMS == 11

n_orig_imgs    = NSTIMS * NDIRS      # 88
n_shifted_imgs = n_orig_imgs * trial_len  # 3256
print(f'n_orig_imgs={n_orig_imgs}, n_shifted_imgs={n_shifted_imgs}')

Using device: cpu
n_orig_imgs=88, n_shifted_imgs=3256


In [29]:
########## LOAD MODEL + REGISTER HOOKS ##########

model = r2plus1d_18(weights=R2Plus1D_18_Weights.KINETICS400_V1)
model.eval()
model = model.to(device)

activation_outputs = {}

def make_hook(layer_name):
    def hook(module, input, output):
        activation_outputs[layer_name] = output.detach().cpu()
    return hook

for layer_name in LAYERS:
    getattr(model, layer_name).register_forward_hook(make_hook(layer_name))

# Warm-up pass to determine output shapes per layer (B, C, T', H', W')
dummy = torch.zeros(1, 3, trial_len, R2PLUS1D_SIZE, R2PLUS1D_SIZE, device=device)
with torch.no_grad():
    _ = model(dummy)

layer_shapes = {}  # layer -> (C, T', H', W')
for layer_name in LAYERS:
    _, C, Tp, Hp, Wp = activation_outputs[layer_name].shape
    layer_shapes[layer_name] = (C, Tp, Hp, Wp)
    print(f'  {layer_name}: C={C}, T\u0027={Tp}, H\u0027={Hp}, W\u0027={Wp}')

  layer1: C=64, T'=37, H'=56, W'=56
  layer2: C=128, T'=19, H'=28, W'=28
  layer3: C=256, T'=10, H'=14, W'=14
  layer4: C=512, T'=5, H'=7, W'=7


In [30]:
########## LOAD OPTICAL-FLOW STIMULI ##########

flow_datasets = createFlowDataset(
    categories, topdir, mydirs,
    orig_shape=orig_shape, input_shape=input_shape,
    scl_factor=scl_factor, N_INSTANCES=N_INSTANCES,
    trial_len=trial_len, stride=1,
)

for insti, arr in flow_datasets.items():
    print(f'Instance {insti}: shape={arr.shape}')
assert flow_datasets[0].shape[0] == n_shifted_imgs

*INSTANCE 0 ...........
*INSTANCE 1 ...........
*INSTANCE 2 ...........
Instance 0: shape=(3256, 12544)
Instance 1: shape=(3256, 12544)
Instance 2: shape=(3256, 12544)


In [31]:
########## PREPROCESSING HELPER ##########

def preprocess_clip(flat_trial, orig_h, orig_w, target_size):
    """Convert one 37-frame trial (37, H*W) uint8 array → normalised RGB video tensor.

    Args:
        flat_trial  : (T, H*W) uint8 numpy array for one orig image
        orig_h, orig_w : spatial dimensions of each raveled frame
        target_size : square side length for R(2+1)D input

    Returns:
        torch.Tensor of shape (1, 3, T, target_size, target_size)
    """
    #print(flat_trial.shape[1]/144)
    T = flat_trial.shape[0]
    frames = flat_trial.reshape(T, orig_h, orig_w).astype('float32') / 255.0  # (T, H, W)
    frames_rgb = np.stack([frames, frames, frames], axis=1)                    # (T, 3, H, W)
    frames_norm = (frames_rgb - KINETICS_MEAN[:, None, None]) / KINETICS_STD[:, None, None]
    t = torch.tensor(frames_norm)  # (T, 3, H, W)
    t = t.permute(1, 0, 2, 3).unsqueeze(0)  # (1, 3, T, H, W)
    t = F.interpolate(t, size=(T, target_size, target_size), mode='trilinear', align_corners=False)
    return t

In [32]:
########## FORWARD PASSES (all layers in parallel) ##########

orig_h, orig_w = input_shape  # 144, 256

# Accumulate activations across instances; shape per layer: (n_orig_imgs, C, T', H', W')
layer_outputs = {
    lname: np.zeros((n_orig_imgs, *layer_shapes[lname]), dtype='float32')
    for lname in LAYERS
}

for insti in range(N_INSTANCES):
    extX = flow_datasets[insti]  # (n_shifted_imgs, H*W) = (3256, H*W)
    # Reshape to (n_orig_imgs, trial_len, H*W)
    extX_clips = extX.reshape(n_orig_imgs, trial_len, -1)  # (88, 37, H*W)
    print(f'Instance {insti}', flush=True)
    t0 = time()

    for img_idx in range(n_orig_imgs):
        print(img_idx, end=' ', flush=True)
        clip_flat = extX_clips[img_idx]  # (37, H*W)
        clip_t = preprocess_clip(clip_flat, orig_h, orig_w, R2PLUS1D_SIZE).to(device)
        # clip_t: (1, 3, 37, 112, 112)

        with torch.no_grad():
            _ = model(clip_t)

        for lname in LAYERS:
            # activation_outputs[lname]: (1, C, T', H', W')
            layer_outputs[lname][img_idx] += activation_outputs[lname].squeeze(0).numpy()

    print(f'  done in {time()-t0:.1f}s', flush=True)

for lname in LAYERS:
    layer_outputs[lname] /= N_INSTANCES
    C, Tp, Hp, Wp = layer_shapes[lname]
    print(f'{lname}: {layer_outputs[lname].shape}  '
          f'min={layer_outputs[lname].min():.3f}  max={layer_outputs[lname].max():.3f}')

Instance 0
0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87   done in 37.9s
Instance 1
0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87   done in 37.8s
Instance 2
0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87   done in 38.5s
layer1: (88, 64, 37, 56, 56)  min=0.000  max=6.718
layer2: (88, 128, 19, 28, 28)  min=0.000  max=10.392
layer3: (88, 256, 10, 14, 14)  min=0.000  max=5.241
l

In [33]:
########## SAMPLE NEURONS + BUILD TENSOR4D + SAVE (per layer) ##########

os.makedirs('../data/sampled', exist_ok=True)

for lname in LAYERS:
    print(f'\n=== {lname} ===')
    C, Tp, Hp, Wp = layer_shapes[lname]
    lo = layer_outputs[lname]  # (n_orig_imgs, C, T', H', W')

    # Reshape to (n_orig_imgs, C, H'*W', T') for consistent sampling with nb09
    lo_t = lo.transpose(0, 1, 3, 4, 2)  # (n_orig_imgs, C, H', W', T')
    lo_t = lo_t.reshape(n_orig_imgs, C, Hp * Wp, Tp)  # (n_orig_imgs, C, H'*W', T')
    n_neurons_per_fmap = Hp * Wp

    # Per-fmap statistics for sampling weights
    all_neurons_maxs  = lo_t.max(axis=(0, 3))   # (C, H'*W')
    all_neurons_means = lo_t.mean(axis=(0, 3))  # (C, H'*W')

    # -- Sample feature maps (maxFr) --
    np.random.seed(seed)
    nfmaps = C
    maxsmean = all_neurons_maxs.mean(1)                  # (C,)
    nonzero_fmaps = int((~np.isclose(maxsmean, 0)).sum())
    n_fmaps_ = min(n_fmaps_to_sample, nonzero_fmaps)
    probs_fmap = maxsmean / maxsmean.sum()
    top_fmaps = np.random.choice(nfmaps, n_fmaps_, replace=False, p=probs_fmap)

    # -- Sample spatial positions within each selected fmap (maxNr) --
    samps_per_fmap = min(samples_per_fmap, n_neurons_per_fmap)
    sampled_neurons = []
    for fi in top_fmaps:
        neuron_vals = all_neurons_maxs[fi]               # (H'*W',)
        nonzero_n   = int((~np.isclose(neuron_vals, 0)).sum())
        samps       = min(samps_per_fmap, nonzero_n)
        probs_n     = neuron_vals / neuron_vals.sum()
        top_nis     = np.random.choice(n_neurons_per_fmap, samps, replace=False, p=probs_n)
        sampled_neurons += list(fi * n_neurons_per_fmap + top_nis)
    sampled_neurons = np.array(sampled_neurons)
    n_neurons_to_pick = len(sampled_neurons)
    print(f'  Sampled {n_neurons_to_pick} neurons from {n_fmaps_} feature maps  (T\u0027={Tp})')

    # -- Build tensor4d (N, NSTIMS, NDIRS, T') --
    tensorX      = np.zeros((n_neurons_to_pick, NSTIMS, NDIRS, Tp), dtype='float32')
    neurons_used = np.empty((n_neurons_to_pick, 3), dtype='int')  # (fmap_idx, h_pos, w_pos)

    for nii, ni in enumerate(sampled_neurons):
        fi   = ni // n_neurons_per_fmap
        posi = ni %  n_neurons_per_fmap
        hi   = posi // Wp
        wi   = posi %  Wp
        neurons_used[nii] = [fi, hi, wi]
        for cati in range(NSTIMS):
            pst = lo_t[cati * NDIRS:(cati + 1) * NDIRS, fi, posi, :]  # (NDIRS, T')
            tensorX[nii, cati] = pst

    print(f'  tensorX shape: {tensorX.shape}')

    # -- Save --
    SUFFIX = f'r2plus1d_{lname}_i{N_INSTANCES}_n{n_neurons_to_pick}_seed{seed}'
    out_tensor  = f'../data/sampled/tensor4d_{SUFFIX}.npy'
    out_neurons = f'../data/sampled/neurons_used_{SUFFIX}.npy'

    if os.path.exists(out_tensor):
        print(f'  [SKIP] {out_tensor} already exists — delete to regenerate')
    else:
        np.save(out_tensor, tensorX)
        np.save(out_neurons, neurons_used)
        print(f'  Saved {out_tensor}')
        print(f'  Saved {out_neurons}')


=== layer1 ===
  Sampled 2000 neurons from 40 feature maps  (T'=37)
  tensorX shape: (2000, 11, 8, 37)
  [SKIP] ../data/sampled/tensor4d_r2plus1d_layer1_i3_n2000_seed17.npy already exists — delete to regenerate

=== layer2 ===
  Sampled 2000 neurons from 40 feature maps  (T'=19)
  tensorX shape: (2000, 11, 8, 19)
  [SKIP] ../data/sampled/tensor4d_r2plus1d_layer2_i3_n2000_seed17.npy already exists — delete to regenerate

=== layer3 ===
  Sampled 2000 neurons from 40 feature maps  (T'=10)
  tensorX shape: (2000, 11, 8, 10)
  [SKIP] ../data/sampled/tensor4d_r2plus1d_layer3_i3_n2000_seed17.npy already exists — delete to regenerate

=== layer4 ===
  Sampled 1960 neurons from 40 feature maps  (T'=5)
  tensorX shape: (1960, 11, 8, 5)
  [SKIP] ../data/sampled/tensor4d_r2plus1d_layer4_i3_n1960_seed17.npy already exists — delete to regenerate
